# Clasificación landuse con Swin Transformer

Implementación de **Swin Transformer** (*Shifted Window Transformer*, Liu et al.,
2021) en Keras. Swin introduce dos ideas clave frente a un ViT clásico:

1. **Atención por ventanas locales**: cada token solo atiende a los tokens de su
   ventana, lo que reduce el coste de `O(n²)` a `O(n)`.
2. **Ventanas desplazadas**: en capas alternas la ventana se desplaza media
   ventana, permitiendo que la información cruce entre ventanas vecinas.

La arquitectura es jerárquica: `PatchMerging` reduce la resolución espacial por
etapas y duplica la dimensión, igual que una CNN. Se entrena **desde cero** para
comparar un Transformer con la CNN y con DenseNet preentrenada.

Adaptación al problema: entrada 128×128, parches de 8×8, cuatro etapas con
dimensiones (48, 96, 192, 384), profundidades (2, 2, 2, 2), cabezas (2, 4, 8, 16)
y ventanas de 4×4.

El modelo se carga desde `resultados/swin/model.weights.h5` por defecto. Cambia
`FORCE_RETRAIN = True` para entrenarlo de nuevo.

## Configuración

In [1]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras import layers, ops
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import AdamW

import common_landuse as cl

FORCE_RETRAIN = False
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cl.SEED)
np.random.seed(cl.SEED)

I0000 00:00:1794800667.499140   92518 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794800667.595488   92518 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1794800670.513293   92518 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Arquitectura Swin

In [2]:
def window_partition(x, window_size):
    _, height, width, channels = x.shape
    x = ops.reshape(
        x,
        (-1, height // window_size, window_size, width // window_size, window_size, channels),
    )
    x = ops.transpose(x, (0, 1, 3, 2, 4, 5))
    return ops.reshape(x, (-1, window_size, window_size, channels))


def window_reverse(windows, window_size, height, width):
    channels = windows.shape[-1]
    x = ops.reshape(
        windows,
        (-1, height // window_size, width // window_size, window_size, window_size, channels),
    )
    x = ops.transpose(x, (0, 1, 3, 2, 4, 5))
    return ops.reshape(x, (-1, height, width, channels))


class WindowAttention(layers.Layer):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = layers.Dense(dim * 3, use_bias=qkv_bias)
        self.proj = layers.Dense(dim)
        self.dropout = layers.Dropout(dropout)
        self.relative_position_bias_table = self.add_weight(
            shape=((2 * window_size - 1) ** 2, num_heads),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            name="relative_position_bias_table",
        )

        coordinates_h = np.arange(window_size)
        coordinates_w = np.arange(window_size)
        coordinates = np.stack(
            np.meshgrid(coordinates_h, coordinates_w, indexing="ij")
        )
        coordinates_flat = coordinates.reshape(2, -1)
        relative = coordinates_flat[:, :, None] - coordinates_flat[:, None, :]
        relative = relative.transpose([1, 2, 0])
        relative[:, :, 0] += window_size - 1
        relative[:, :, 1] += window_size - 1
        relative[:, :, 0] *= 2 * window_size - 1
        self.relative_position_index = ops.convert_to_tensor(
            relative.sum(-1), dtype="int32"
        )

    def call(self, x, attn_mask=None, training=False):
        tokens = x.shape[1]
        head_dim = self.dim // self.num_heads
        qkv = self.qkv(x)
        qkv = ops.reshape(qkv, (-1, tokens, 3, self.num_heads, head_dim))
        qkv = ops.transpose(qkv, (2, 0, 3, 1, 4))
        query, key, value = qkv[0], qkv[1], qkv[2]

        attention = (query * self.scale) @ ops.transpose(key, (0, 1, 3, 2))
        indices = ops.reshape(self.relative_position_index, (-1,))
        bias = ops.take(self.relative_position_bias_table, indices, axis=0)
        bias = ops.reshape(bias, (tokens, tokens, -1))
        bias = ops.transpose(bias, (2, 0, 1))
        attention = attention + bias[None]

        if attn_mask is not None:
            windows = attn_mask.shape[0]
            attention = ops.reshape(
                attention, (-1, windows, self.num_heads, tokens, tokens)
            )
            attention = attention + attn_mask[None, :, None, :, :]
            attention = ops.reshape(attention, (-1, self.num_heads, tokens, tokens))

        attention = ops.softmax(attention, axis=-1)
        attention = self.dropout(attention, training=training)
        output = attention @ value
        output = ops.transpose(output, (0, 2, 1, 3))
        output = ops.reshape(output, (-1, tokens, self.dim))
        return self.dropout(self.proj(output), training=training)


class SwinBlock(layers.Layer):
    def __init__(
        self,
        dim,
        num_heads,
        input_resolution,
        window_size=4,
        shift_size=0,
        mlp_ratio=4.0,
        dropout=0.0,
        drop_path=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.height, self.width = input_resolution
        self.window_size = min(window_size, self.height, self.width)
        self.shift_size = (
            0
            if self.window_size <= 1
            else min(shift_size, self.window_size // 2)
        )
        self.norm1 = layers.LayerNormalization(epsilon=1e-5)
        self.attn = WindowAttention(
            dim, self.window_size, num_heads, dropout=dropout
        )
        self.norm2 = layers.LayerNormalization(epsilon=1e-5)
        self.mlp = keras.Sequential(
            [
                layers.Dense(int(dim * mlp_ratio)),
                layers.Activation("gelu"),
                layers.Dropout(dropout),
                layers.Dense(dim),
                layers.Dropout(dropout),
            ]
        )
        self.drop_path = layers.Dropout(drop_path)
        self.attn_mask = None
        if self.shift_size > 0:
            height, width = self.height, self.width
            height_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            width_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            mask = np.zeros((1, height, width, 1), dtype=np.float32)
            count = 0
            for height_slice in height_slices:
                for width_slice in width_slices:
                    mask[:, height_slice, width_slice, :] = count
                    count += 1
            mask = ops.convert_to_tensor(mask)
            mask_windows = window_partition(mask, self.window_size)
            mask_windows = ops.reshape(
                mask_windows, (-1, self.window_size * self.window_size)
            )
            attention_mask = mask_windows[:, None, :] - mask_windows[:, :, None]
            self.attn_mask = ops.where(attention_mask != 0, -100.0, 0.0)

    def call(self, x, training=False):
        shortcut = x
        x = self.norm1(x)
        x = ops.reshape(x, (-1, self.height, self.width, self.dim))
        if self.shift_size > 0:
            x = ops.roll(
                x, shift=(-self.shift_size, -self.shift_size), axis=(1, 2)
            )
        windows = window_partition(x, self.window_size)
        windows = ops.reshape(
            windows, (-1, self.window_size * self.window_size, self.dim)
        )
        windows = self.attn(windows, attn_mask=self.attn_mask, training=training)
        windows = ops.reshape(
            windows, (-1, self.window_size, self.window_size, self.dim)
        )
        x = window_reverse(windows, self.window_size, self.height, self.width)
        if self.shift_size > 0:
            x = ops.roll(x, shift=(self.shift_size, self.shift_size), axis=(1, 2))
        x = ops.reshape(x, (-1, self.height * self.width, self.dim))
        x = shortcut + self.drop_path(x, training=training)
        x = x + self.drop_path(self.mlp(self.norm2(x)), training=training)
        return x


class BasicLayer(layers.Layer):
    def __init__(
        self,
        dim,
        depth,
        num_heads,
        input_resolution,
        window_size=4,
        drop_path_rate=0.0,
        dropout=0.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.blocks = [
            SwinBlock(
                dim,
                num_heads,
                input_resolution,
                window_size,
                shift_size=0 if index % 2 == 0 else window_size // 2,
                dropout=dropout,
                drop_path=drop_path_rate * index / max(depth - 1, 1),
            )
            for index in range(depth)
        ]

    def call(self, x, training=False):
        for block in self.blocks:
            x = block(x, training=training)
        return x


class PatchEmbed(layers.Layer):
    def __init__(self, dim, patch_size, input_size, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Conv2D(dim, kernel_size=patch_size, strides=patch_size)
        self.norm = layers.LayerNormalization(epsilon=1e-5)
        self.num_patches = (input_size // patch_size) ** 2
        self.pos_embed = self.add_weight(
            shape=(1, self.num_patches, dim),
            initializer=keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            name="pos_embed",
        )

    def call(self, x):
        x = self.proj(x)
        _, height, width, channels = x.shape
        x = ops.reshape(x, (-1, height * width, channels))
        return self.norm(x + self.pos_embed)


class PatchMerging(layers.Layer):
    def __init__(self, dim, input_resolution, **kwargs):
        super().__init__(**kwargs)
        self.height, self.width = input_resolution
        self.norm = layers.LayerNormalization(epsilon=1e-5)
        self.reduction = layers.Dense(2 * dim, use_bias=False)

    def call(self, x):
        channels = x.shape[-1]
        x = ops.reshape(x, (-1, self.height, self.width, channels))
        top_left = x[:, 0::2, 0::2, :]
        top_right = x[:, 1::2, 0::2, :]
        bottom_left = x[:, 0::2, 1::2, :]
        bottom_right = x[:, 1::2, 1::2, :]
        x = ops.concatenate(
            [top_left, top_right, bottom_left, bottom_right], axis=-1
        )
        x = ops.reshape(
            x, (-1, (self.height // 2) * (self.width // 2), 4 * channels)
        )
        return self.reduction(self.norm(x))


def build_swin(
    input_size=128,
    num_classes=3,
    dims=(48, 96, 192, 384),
    depths=(2, 2, 2, 2),
    heads=(2, 4, 8, 16),
    patch_size=8,
    window_size=4,
    dropout=0.05,
    drop_path_rate=0.0,
):
    inputs = layers.Input(shape=(input_size, input_size, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.Rescaling(1.0 / 255.0)(x)
    x = PatchEmbed(dims[0], patch_size, input_size)(x)

    height = width = input_size // patch_size
    for index, (dim, depth, num_heads) in enumerate(zip(dims, depths, heads)):
        x = BasicLayer(
            dim,
            depth,
            num_heads,
            (height, width),
            window_size,
            drop_path_rate=drop_path_rate,
            dropout=dropout,
        )(x)
        if index < len(dims) - 1:
            x = PatchMerging(dim, (height, width))(x)
            height, width = height // 2, width // 2

    x = layers.LayerNormalization(epsilon=1e-5)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="swin_landuse")

## Datos, entrenamiento y carga desde caché

In [3]:
data = cl.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cl.one_hot(y_train_labels)
y_val = cl.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)
print("Test distribution:", {name: int(np.sum(y_test_labels == name)) for name in cl.CLASSES})

Train: (956, 128, 128, 3) | Validation: (169, 128, 128, 3) | Test: (375, 128, 128, 3)
Test distribution: {'denseresidential': 125, 'harbor': 125, 'river': 125}


In [4]:
def fit_swin(model):
    model.compile(
        loss="categorical_crossentropy",
        optimizer=AdamW(learning_rate=3e-4, weight_decay=0.01),
        metrics=["accuracy"],
    )
    callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=12, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ]
    history, seconds = cl.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=40,
        batch_size=32,
        callbacks=callbacks,
    )
    return model, history, seconds


def evaluate_swin(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cl.predict_keras(model, X_test)
    predicted = np.array(cl.CLASSES)[predicted_idx]
    metrics = cl.compute_metrics(
        name="Swin Transformer",
        family="Transformer",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history.history["loss"]),
        hyperparameters={
            "patch_size": 8,
            "window_size": 4,
            "dims": [48, 96, 192, 384],
            "depths": [2, 2, 2, 2],
            "heads": [2, 4, 8, 16],
            "optimizer": "AdamW",
            "lr": 3e-4,
            "weight_decay": 0.01,
            "label_smoothing": 0.0,
            "batch_size": 32,
            "epochs_max": 40,
            "pretrained": False,
        },
        notes="Swin Transformer trained from scratch (no ImageNet weights).",
    )
    cl.save_results(
        "swin",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=model,
    )
    # The .keras file contains custom layers, so the weights are also saved
    # separately for a reliable reload.
    model.save_weights(cl.result_dir("swin") / cl.ARTIFACTS["weights"])
    return metrics


def load_swin(folder):
    weights_path = cl.result_dir(folder) / cl.ARTIFACTS["weights"]
    if not weights_path.exists():
        return None
    model = build_swin()
    model.load_weights(weights_path)
    return model

In [5]:
model, history, metrics, retrained = cl.load_or_train(
    "swin",
    build_model=build_swin,
    fit_model=fit_swin,
    evaluate_model=evaluate_swin,
    load_model=load_swin,
    force=FORCE_RETRAIN,
)
print("Test accuracy:", round(metrics["accuracy"], 4))

E0000 00:00:1794800680.640383   92518 cuda_executor.cc:1690] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:68

W0000 00:00:1794800680.640996   92580 cuda_executor.cc:1710] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:77
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:107
external/xla/xla/stream_executor/cuda/cudnn_api_wrappers.cc:112

W0000 00:00:1794800680.656244   92518 dlopen_checker.cc:38] Could not load dynamic library 'libcudart.so.12'; dlerror: libcudart.so.12: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/andres/Deskt

[cache] Using trained model from /home/andres/Desktop/vision-por-computador/guia-4/reto-2/resultados/swin
Test accuracy: 0.9653


## Observaciones

* Swin es una arquitectura **jerárquica** con atención local; a diferencia de un
  ViT plano, captura estructuras de varias escalas, algo útil en escenas aéreas
  con edificios, agua y costa.
* Al entrenarse **desde cero** con 956 imágenes, el Transformer dispone de muchos
  menos datos que un Swin preentrenado en ImageNet. La augmentación, el weight
  decay y el early stopping son imprescindibles para controlar el sobreajuste.
* La comparación con DenseNet separa dos efectos: **arquitectura** (atención vs
  convolución) y **preentrenamiento** (ImageNet vs desde cero).
* El modelo se guarda como `model.keras` y también como `model.weights.h5`; la
  celda `load_swin` reconstruye la arquitectura y carga los pesos para evitar
  problemas de deserialización de las capas personalizadas.